In [1]:
from collections import defaultdict

In [3]:
class FSA:
    def __init__(self):
        self.transitions = defaultdict(dict)
        self.start_state = None
        self.final_states = set()

    def add_transition(self, from_state, symbol, to_state):
        self.transitions[from_state][symbol] = to_state

    def set_start(self, state):
        self.start_state = state

    def add_final(self, state):
        self.final_states.add(state)

    def accepts(self, word):
        state = self.start_state
        for ch in word:
            if ch not in self.transitions[state]:
                return False
            state = self.transitions[state][ch]
        return state in self.final_states


In [5]:
def build_noun_fsa():
    fsa = FSA()
    fsa.set_start(0)

    words = ["cat", "dog", "fox"]

    state = 1
    for word in words:
        curr = 0
        for ch in word:
            fsa.add_transition(curr, ch, state)
            curr = state
            state += 1

        # singular
        fsa.add_final(curr)

        # plural
        if word == "fox":
            fsa.add_transition(curr, 'e', state)
            fsa.add_transition(state, 's', state+1)
            fsa.add_final(state+1)
            state += 2
        else:
            fsa.add_transition(curr, 's', state)
            fsa.add_final(state)
            state += 1

    return fsa

noun_fsa = build_noun_fsa()


In [7]:
def build_verb_fsa():
    fsa = FSA()
    fsa.set_start(0)

    roots = ["walk", "jump"]
    state = 1

    for root in roots:
        curr = 0
        for ch in root:
            fsa.add_transition(curr, ch, state)
            curr = state
            state += 1

        # base verb
        fsa.add_final(curr)

        # walks / jumps
        fsa.add_transition(curr, 's', state)
        fsa.add_final(state)
        state += 1

        # walked / jumped
        fsa.add_transition(curr, 'e', state)
        fsa.add_transition(state, 'd', state+1)
        fsa.add_final(state+1)
        state += 2

        # walking / jumping
        fsa.add_transition(curr, 'i', state)
        fsa.add_transition(state, 'n', state+1)
        fsa.add_transition(state+1, 'g', state+2)
        fsa.add_final(state+2)
        state += 3

    return fsa

verb_fsa = build_verb_fsa()


In [9]:
test_words = ["cats", "foxes", "foxs", "walking", "walkes"]

for w in test_words:
    result = noun_fsa.accepts(w) or verb_fsa.accepts(w)
    print(f"{w:10} → {'ACCEPTED' if result else 'REJECTED'}")


cats       → ACCEPTED
foxes      → ACCEPTED
foxs       → REJECTED
walking    → ACCEPTED
walkes     → REJECTED


In [11]:
def stem_suffix(word):
    noun_roots = ["cat", "dog", "fox"]
    verb_roots = ["walk", "jump"]

    for root in noun_roots + verb_roots:
        if word.startswith(root):
            return root, word[len(root):]
    return None, None


In [13]:
def classify_word(word):
    if noun_fsa.accepts(word):
        stem, suffix = stem_suffix(word)
        return f"{word} → NOUN → {stem} + {suffix}"
    elif verb_fsa.accepts(word):
        stem, suffix = stem_suffix(word)
        return f"{word} → VERB → {stem} + {suffix}"
    else:
        return f"{word} → INVALID"


In [15]:
words = ["cat", "cats", "foxs", "walked", "jumping", "jumpings"]

for w in words:
    print(classify_word(w))


cat → NOUN → cat + 
cats → NOUN → cat + s
foxs → INVALID
walked → VERB → walk + ed
jumping → VERB → jump + ing
jumpings → INVALID


In [17]:
invalid_words = ["foxs", "walkes", "jumpings"]

for w in invalid_words:
    print(f"{w} → {'REJECTED' if not (noun_fsa.accepts(w) or verb_fsa.accepts(w)) else 'ACCEPTED'}")


foxs → REJECTED
walkes → REJECTED
jumpings → REJECTED
